In [4]:
import torch
from src.models.architecture import UnlaminatedCopyDetector

model_path = "src/models/best_cnn_detector_correctedv2.pth"

# get all weights and sum all weights
def load_model():
    model = UnlaminatedCopyDetector()
    model.load_state_dict(
        {
            k.replace("module.", ""): v
            for k, v in torch.load(model_path, map_location="cpu").items()
        }
    )
    model.eval()
    return model

def get_weights_sum(model_path):
    """Calculate the sum of all weights in a .pth file"""
    state_dict = torch.load(model_path, map_location="cpu")
    
    total_sum = 0.0
    for key, tensor in state_dict.items():
        # Remove 'module.' prefix if present
        clean_key = key.replace("module.", "")
        total_sum += tensor.sum().item()
        print(f"{clean_key}: {tensor.sum().item()}")
    
    return total_sum

# Usage
total = get_weights_sum(model_path)
print(f"\nTotal sum of all weights: {total}")

conv1.weight: -0.3822031617164612
conv1.bias: 0.8681186437606812
conv2.weight: 1.856507658958435
conv2.bias: 0.02035859227180481
conv3.weight: 20.353849411010742
conv3.bias: 0.01855941116809845
conv4.weight: 16.424667358398438
conv4.bias: 0.33567172288894653
conv5.weight: 18.087818145751953
conv5.bias: 0.6700835227966309
conv6.weight: 33.90116882324219
conv6.bias: -0.05226639658212662
conv7.weight: 7.504979610443115
conv7.bias: -0.11804032325744629
fc.weight: -0.4189821481704712
fc.bias: 0.05544518679380417

Total sum of all weights: 99.12573605775833


In [ ]:
from torchvision import transforms
from PIL import Image

model_path = "src/models/best_cnn_detector_correctedv2.pth"
IMAGE_SIZE = 76


# Load model
def load_model():
    model = UnlaminatedCopyDetector()
    model.load_state_dict(
        {
            k.replace("module.", ""): v
            for k, v in torch.load(model_path, map_location="cpu").items()
        }
    )
    model.eval()
    return model


# Define transform - use Grayscale (1 channel)
transform = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
)


# Run inference
def predict_image(image_path, threshold=0.5):
    model = load_model()
    device = torch.device("cpu")
    model = model.to(device)

    # Load and transform image - convert to grayscale
    image = Image.open(image_path)
    tensor = transform(image).unsqueeze(0).to(device)

    # Make prediction
    with torch.inference_mode():
        logits = model(tensor)
        prob_unlaminated = torch.sigmoid(logits).item()

    # Determine prediction
    prediction = (
        "UNLAMINATED (SPOOF)" if prob_unlaminated > threshold else "LAMINATED (GENUINE)"
    )
    confidence = (
        prob_unlaminated if prob_unlaminated > threshold else 1 - prob_unlaminated
    )

    return prediction, prob_unlaminated, confidence


# Test
image_path = r"C:\Users\USER\Downloads\ktp-test.jpeg"
prediction, probability, confidence = predict_image(image_path)

print(f"Prediction: {prediction}")
print(f"Probability (unlaminated): {probability:.4f}")
print(f"Confidence: {confidence:.2%}")